In [ ]:
# imports

import pandas as pd
import numpy as np
import json
import polars as pl
from pathlib import Path
from datasets import Dataset, DatasetDict, concatenate_datasets
import torch
from torch.utils.data import DataLoader
import glob
import os
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (set_seed,
                          TrainingArguments,
                          Trainer,
                          Qwen3ForSequenceClassification,
                          AutoTokenizer,
                          DataCollatorWithPadding)
from tqdm import tqdm
from peft import LoraConfig, get_peft_model, TaskType
import shutil
from google.colab import files
import gc

In [ ]:
CURRENT_SEED = 42

set_seed(CURRENT_SEED)

In [ ]:
# Extract tabfact zip if notebook is in google colab

import zipfile

zip_path = '/content/tabfact_data.zip'
extract_path = '/content/'

print("Unzipping dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Unzipping completed! Files extracted to:", extract_path)

In [ ]:
# Set data path, if notebook is in google colab use content, if not it is in the directory above

# # Local repository path
# DATA_PATH = "../tabfact_data"

# Colab path
DATA_PATH = "/content/tabfact_data"

In [ ]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, label, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, label in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [ ]:
# Load in table dataframe

# # DOESN'T WORK IN COLAB BUT FASTER IN LOCAL REPO
# df_tables = (
#     pl.read_csv(
#         "../tabfact_data/tables/*.csv",
#         has_header=False,
#         separator="\1", # Non-existent delimiter so full line/file stays intact
#         new_columns=["table_content"],
#         include_file_paths="file_name",
#     )
#     .group_by("file_name", maintain_order=True) # Group all the table lines together
#     .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
#     .to_pandas()
# )

csv_files = sorted(glob.glob(f"{DATA_PATH}/tables/*.csv"))

df_tables = (
    pl.concat([
        pl.read_csv(
            file,
            has_header=False,
            separator="\1",
            new_columns=["table_content"],
        ).with_columns(pl.lit(Path(file).name).alias("file_name"))
        for file in csv_files
    ])
    .group_by("file_name", maintain_order=True)  # Group all the table lines together
    .agg(pl.col("table_content").str.join("\n"))  # Concatenate table lines together
    .to_pandas()
)

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

In [ ]:
# Defining column labelled rows
print(df_tables.shape)

df_tables["headers"] = df_tables["table_content"].apply(lambda x: x.split("\n")[0].split("#")) # Get the first line of the table content as headers

df_tables["rows"] = df_tables["table_content"].apply(lambda x: [row.split("#") for row in x.split("\n")[1:]]) # Get the rest of the lines as rows

df_tables["column_labelled_rows"] = df_tables.apply(
    lambda row: "\n".join(
        ", ".join(f"{h}: {v}" for h, v in zip(row["headers"], r))
        for r in row["rows"]
    ),
    axis=1,
)

print(df_tables["column_labelled_rows"].iloc[0])

In [ ]:
# Load in non table data frames

df_full_cleaned = load_and_convert_data_to_dataframe(f"{DATA_PATH}/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['label'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['label'] == 1]

df_r1 = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r1_training_all.json")
df_r2 = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r2_training_all.json")


df_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/train_examples.json")
df_test = load_and_convert_data_to_dataframe(f"{DATA_PATH}/test_examples.json")
df_validation = load_and_convert_data_to_dataframe(f"{DATA_PATH}/val_examples.json")


In [ ]:
def get_table_claim_df(df, table_df_col, table_before_claim=True, table_prefix="TABLE: ", claim_prefix="CLAIM: ", table_claim_separator=" | ", before_evidence_text=""):
    df_table_claim = pd.merge(
        df_tables[["file_name", table_df_col]],
        df[["table_id", "statement", "label"]],
        left_on="file_name",
        right_on="table_id",
        how="inner"
    ).reset_index(drop=True)
    if table_before_claim:
      before_evidence_text += "Here is the TABLE and then the CLAIM: \n" if before_evidence_text != "" else ""
      df_table_claim["table_claim"] = (
        before_evidence_text +
        table_prefix +
        df_table_claim[table_df_col] +
        table_claim_separator +
        claim_prefix +
        df_table_claim["statement"]
      )
    else:
        before_evidence_text += "Here is the CLAIM and then the TABLE: \n" if before_evidence_text != "" else ""
        df_table_claim["table_claim"] = (
          before_evidence_text +
          claim_prefix +
          df_table_claim["statement"] +
          table_claim_separator +
          table_prefix +
          df_table_claim[table_df_col]
        )

    return df_table_claim[["table_claim", "label"]].copy()

In [ ]:
df_train_table_claim = get_table_claim_df(df_training, "table_content", table_before_claim=False, table_claim_separator="\n")
df_val_table_claim = get_table_claim_df(df_validation, "table_content", table_before_claim=False, table_claim_separator="\n")
df_test_table_claim = get_table_claim_df(df_test, "table_content", table_before_claim=False, table_claim_separator="\n")
df_r1_table_claim = get_table_claim_df(df_r1, "table_content", table_before_claim=False, table_claim_separator="\n")
df_r2_table_claim = get_table_claim_df(df_r2, "table_content", table_before_claim=False, table_claim_separator="\n")

print(df_val_table_claim["table_claim"].iloc[0])

print(df_val_table_claim.shape)

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
qwen_tokenizer.padding_side = "right"
qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

base_qwen_model = Qwen3ForSequenceClassification.from_pretrained(
    "Qwen/Qwen3-0.6B",
    num_labels=2,
    torch_dtype=torch.float16,
    device_map="auto"
)

base_qwen_model.config.pad_token_id = qwen_tokenizer.pad_token_id

In [ ]:
# Lora config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

qwen_peft_model = get_peft_model(base_qwen_model, lora_config)

In [ ]:
def get_reduced_random_test_dataset(df, size):
  entailed_test_dataset = Dataset.from_pandas(df[df['label'] == 1])
  refuted_test_dataset = Dataset.from_pandas(df[df['label'] == 0])

  generator = np.random.default_rng(CURRENT_SEED)
  entailed_random_indices = generator.choice(len(entailed_test_dataset), size=int(size/2), replace=False)
  refuted_random_indices = generator.choice(len(refuted_test_dataset), size=int(size/2), replace=False)

  return concatenate_datasets([
      entailed_test_dataset.select(entailed_random_indices),
      refuted_test_dataset.select(refuted_random_indices)
  ])

In [ ]:
def preprocess_batch(entries):
    return qwen_tokenizer(entries["table_claim"], max_length=768, truncation=True)

train_dataset = Dataset.from_pandas(df_train_table_claim)
val_dataset = Dataset.from_pandas(df_val_table_claim)
test_dataset = Dataset.from_pandas(df_test_table_claim)

datasets = DatasetDict({
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset,
})

tokenized_datasets = datasets.map(preprocess_batch, batched=True, remove_columns="table_claim")

# pad_to_multiple_of=8 enables fast Tensor Core execution on the L4 GPU
data_collator = DataCollatorWithPadding(tokenizer=qwen_tokenizer, pad_to_multiple_of=8)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(y_true=labels, y_pred=predictions)
    f1 = f1_score(y_true=labels, y_pred=predictions)
    precision = precision_score(y_true=labels, y_pred=predictions)
    recall = recall_score(y_true=labels, y_pred=predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
def test_on_test_set():
    tokenized_test_dataset = tokenized_datasets["test"]

    test_results = trainer.predict(tokenized_test_dataset)

    print("Test Set Performance")
    for metric_name, metric_value in test_results.metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
# Check attention mask lengths safely across inhomogeneous sequence lengths

# Keep the format as standard python lists
python_dataset = tokenized_datasets["train"].with_format(None)
train_attention_masks = python_dataset["attention_mask"]

# Sum each individual list's mask values (1s)
true_lengths = [sum(mask) for mask in train_attention_masks]

lengths_series = pd.Series(true_lengths)

print("--- Training Set Token Length Summary ---")
print(lengths_series.describe())

max_limit_count = (lengths_series == 768).sum()
print(
    f"\nNumber of samples at 768 tokens (truncated): "
    f"{max_limit_count} out of {len(lengths_series)} "
    f"({max_limit_count / len(lengths_series) * 100:.2f}%)"
)

In [ ]:
# Prevent memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clear memory
gc.collect()
torch.cuda.empty_cache()

In [ ]:
if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")

    # optimised for L4 gpu
    num_train_epochs = 1
    batch_size = 32
    grad_accum = 1
    steps_per_epoch = len(tokenized_datasets['train']) // (batch_size * grad_accum)
    total_steps = steps_per_epoch * num_train_epochs
    warmup_steps = int(0.05 * total_steps)

    training_args = TrainingArguments(
        output_dir='./tabfact_qwen3_lora_results',
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        learning_rate=2e-4,
        seed=CURRENT_SEED,
        data_seed=CURRENT_SEED,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        bf16=True,
        # optimisation for faster execution
        gradient_checkpointing=True, # saves memory to allow higher batch sizes
        optim="adamw_torch_fused",
        dataloader_num_workers=4,
        dataloader_pin_memory=True
    )

    # Ensure model gradients are handled standardly
    qwen_peft_model.config.use_cache = False

    trainer = Trainer(
        model=qwen_peft_model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['val'],
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model("./tabfact_qwen3_lora")

    test_on_test_set()
else:
    print("GPU NOT active. Running on CPU. Training will not commence")

In [ ]:
# Zip model and tokenizer together

print("Saving tokenizer to './tabfact_qwen3_lora'...")
qwen_tokenizer.save_pretrained("./tabfact_qwen3_lora")

archive_name = "/content/tabfact_qwen3_lora_model"
print(f"Zipping './tabfact_qwen3_lora' to '{archive_name}.zip'...")
shutil.make_archive(archive_name, 'zip', "./tabfact_qwen3_lora")
print("Zipping complete! You can download 'tabfact_qwen3_lora_model.zip' directly from the Colab file pane on the left.")

In [ ]:
# Automatic browser download of the zip file
files.download('/content/tabfact_qwen3_lora_model.zip')